#### Author's Model

In [1]:
import torch
from torchinfo import summary
from core.diffuser.networks.embeddUnet import ConditionalUnet1D
from core.diffuser.config.diffuser_v1 import DiffuserV1Config

In [2]:
config = DiffuserV1Config()

net = ConditionalUnet1D(
        input_dim=config.network_config["unet_config"]["action_dim"],
        diffusion_step_embed_dim=config.network_config["unet_config"]["diffusion_step_embed_dim"],
        global_cond_dim=config.network_config["vit_config"]["num_classes"] + 
                        config.network_config["mlp_config"]["embed_dim"],
        network_config=config.network_config,
        is_cnn=config.is_CNN
    )
device = 'cuda'
net = net.to(device)


In [4]:
# Non-RL Adapted Diffuser Model from yulin-li github: about 80.79M parameter size

from torchinfo import summary

B = 64
sample = torch.randn(B, 32, 2).to(device)
timestep = torch.tensor([1]*B).long().to(device)
map_cond = torch.randn(B, 1, 8, 8).to(device)
env_cond = torch.randn(B, 4).to(device)

stats = summary(
    net,
    input_data={'sample': sample, 'timestep': timestep, 'map_cond': map_cond, 'env_cond': env_cond},
    depth=1,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("output_size", "num_params", "params_percent", "mult_adds"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                            Output Shape              Param #                   Param %                   Mult-Adds
ConditionalUnet1D (ConditionalUnet1D)                        [64, 32, 2]               --                             --                   --
├─ViT (map_encoder): 1-1                                     [64, 32]                  12,637,760                 15.64%                   808,620,032
├─MLP (env_cond): 1-2                                        [64, 64]                  83,520                      0.10%                   5,345,280
├─Sequential (diffusion_step_encoder): 1-3                   [64, 256]                 525,568                     0.65%                   33,636,352
├─ModuleList (down_modules): 1-4                             --                        28,862,016                 35.72%                   --
├─ModuleList (mid_modules): 1-5                              --                        22,678,208                 28.

#### Current VSSM + Attention based Map Encoder + Conv1D Diffusion Decoder with FiLM

In [1]:
import torch

from core.datasets import plane_dataset_embeed # Dataset
from core.networks.traj_planner.TrajectoryPlanner import TrajectoryPlanner # Model Class
from config.trajectory_planner import TrajectoryPlannerConfig # Configuration Class

# trainer for training network
from core.trainer.plane_diffusion_trainer_embeed import PlaneDiffusionTrainer # Trainer
from core.networks.utils.enums import UPSampleStyle

/exhdd/seungyu/.conda/envs/diffusion_motion/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/exhdd/seungyu/.conda/envs/diffusion_motion/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/exhdd/seungyu/.conda/envs/diffusion_motion/lib/python3.12/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [2]:
traj_planner_config = TrajectoryPlannerConfig()

traj_planner_dict = traj_planner_config.to_dict()
traj_planner_net_config = traj_planner_dict['network_config']
device='cuda'
net = TrajectoryPlanner(traj_planner_net_config).to(device)

In [8]:
from torchinfo import summary
B=64
# sample: torch.Tensor,
# timestep: Union[torch.Tensor, float, int],
# map_cond=None,
# env_cond=None
sample = torch.randn(B, 32, 2).to(device)
timestep = torch.tensor([0.5]*B).to(device)
map_cond = torch.randn(B, 3, 8, 8).float().to(device)
env_cond = torch.randint(0, 7, (B, 4)).float().to(device)

stats = summary(
    net,
    input_data={'sample': sample, 'timestep': timestep, 'map_cond': map_cond, 'env_cond': env_cond},
    depth=1,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("output_size", "num_params", "params_percent", "mult_adds"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                                                Output Shape              Param #                   Param %                   Mult-Adds
TrajectoryPlanner (TrajectoryPlanner)                                            [64, 32, 2]               --                             --                   --
├─VSSMapEncoder (map_encoder): 1-1                                               [64, 256, 8, 8]           11,815,424                 15.32%                   45,020,545,024
├─OBSEmbedEncoder (env_encoder): 1-2                                             [64, 128]                 157,888                     0.20%                   10,170,368
├─FeatUpsampler (map_upsampler): 1-3                                             [64, 256, 8, 8]           --                             --                   --
├─DiffusionTrajDecoder (diff_decoder): 1-4                                       [64, 32, 2]               65,145,730                 84.47%                   26,1

### Diffuser V2

In [1]:
import torch
from core.diffuser.networks.traj_planner.TrajectoryPlanner import TrajectoryPlanner # Model Class
from core.diffuser.config.diffuser_v2 import DiffuserV2Config # Configuration Class
from core.diffuser.config.DDP.diffuser_ddp import DiffuserDDPConfig

In [2]:
# traj_planner_config = DiffuserV2Config()
traj_planner_config = DiffuserDDPConfig()

traj_planner_dict = traj_planner_config.to_dict()
traj_planner_net_config = traj_planner_dict['network_config']
device='cuda:3'
net = TrajectoryPlanner(traj_planner_net_config).to(device)

In [3]:
from torchinfo import summary
B=64

# sample: torch.Tensor,
# timestep: Union[torch.Tensor, float, int],
# map_cond=None,
# env_cond=None
sample = torch.randn(B, 128, 2).to(device)
timestep = torch.tensor([1.0]*B).to(device)
map_cond = torch.randn(B, 1, 32, 32).float().to(device)
env_cond = torch.randint(0, 32, (B, 4)).float().to(device)

stats = summary(
    net,
    input_data={'sample': sample, 'timestep': timestep, 'map_cond': map_cond, 'env_cond': env_cond},
    depth=1,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("output_size", "num_params", "params_percent", "mult_adds"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                                 Output Shape              Param #                   Param %                   Mult-Adds
TrajectoryPlanner (TrajectoryPlanner)                             [64, 128, 2]              --                             --                   --
├─ViT (map_encoder): 1-1                                          [64, 128, 32, 32]         14,960,496                 28.28%                   9,404,863,488
├─OBSEmbedEncoder (env_encoder): 1-2                              [64, 128]                 25,152                      0.05%                   1,609,728
├─DiffusionTrajDecoder (diff_decoder): 1-3                        [64, 128, 2]              37,924,674                 71.68%                   44,603,977,728
Total params: 52,910,322
Trainable params: 52,910,322
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 54.01
Input size (MB): 0.33
Forward/backward pass size (MB): 2491.42
Params size (MB): 211.51
Estimated Total Size 

#### Adapted Diffuser V2 + Bi-Latent Fusion

In [1]:
import torch
from core.diffuser.networks.traj_planner.BiLatentTrajectoryPlanner import BiLatentTrajectoryPlanner # Model Class
from core.diffuser.config.diffuser_bil import BiLatentTrajectoryPlannerConfig # Configuration Class
from core.diffuser.config.DDP.diffuser_bil_ddp import DiffuserBilDDPConfig

In [2]:
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
# traj_planner_config = BiLatentTrajectoryPlannerConfig()
traj_planner_config = DiffuserBilDDPConfig()


traj_planner_dict = traj_planner_config.to_dict()
traj_planner_net_config = traj_planner_dict['network_config']

device='cuda:3'

net = BiLatentTrajectoryPlanner(traj_planner_net_config)
noise_scheduler = DDPMScheduler(
            num_train_timesteps=100,
            beta_schedule='squaredcos_cap_v2',
            clip_sample=True,
            prediction_type='epsilon')
net.set_noise_scheduler(noise_scheduler)
net = net.to(device)

In [3]:
from torchinfo import summary
B=64
# sample: torch.Tensor,
# timestep: Union[torch.Tensor, float, int],
# map_cond=None,
# env_cond=None
sample = torch.randn(B, 128, 2).to(device)
timestep = torch.tensor([1]*B).long().to(device)
map_cond = torch.randn(B, 1, 32, 32).float().to(device)
env_cond = torch.randint(0, 32, (B, 4)).float().to(device)

stats = summary(
    net,
    input_data={'sample': sample, 'timestep': timestep, 'map_cond': map_cond, 'env_cond': env_cond},
    depth=2,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("output_size", "num_params", "params_percent", "mult_adds"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                                 Output Shape              Param #                   Param %                   Mult-Adds
BiLatentTrajectoryPlanner (BiLatentTrajectoryPlanner)             [64, 128, 2]              --                             --                   --
├─ViT (map_encoder): 1-1                                          [64, 128, 32, 32]         33,792                      0.06%                   --
│    └─Sequential (stem_conv): 2-1                                [64, 32, 32, 32]          9,568                       0.02%                   627,048,448
│    └─Sequential (to_patch_embedding): 2-2                       [64, 64, 512]             2,104,832                   3.90%                   134,709,248
│    └─Dropout (dropout): 2-3                                     [64, 65, 512]             --                             --                   --
│    └─TransformerEncoder (transformer): 2-4                      [64, 65, 512]             1

#### Diffuser V2 + Multi-Hypothesis Refiner

In [12]:
import torch
from core.diffuser.networks.traj_planner.MultiHypothesisTrajectoryPlanner import MultiHypothesisTrajectoryPlanner # Model Class
from core.diffuser.config.diffuser_mhypo import MultiHypothesisTrajectoryPlannerConfig # Configuration Class

In [13]:
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
traj_planner_config = MultiHypothesisTrajectoryPlannerConfig()


traj_planner_dict = traj_planner_config.to_dict()
traj_planner_net_config = traj_planner_dict['network_config']

device='cuda'

net = MultiHypothesisTrajectoryPlanner(traj_planner_net_config)
noise_scheduler = DDPMScheduler(
            num_train_timesteps=100,
            beta_schedule='squaredcos_cap_v2',
            clip_sample=True,
            prediction_type='epsilon')
net.set_noise_scheduler(noise_scheduler)
net = net.to(device)

In [14]:
from torchinfo import summary
B=64

# sample: torch.Tensor,
# timestep: Union[torch.Tensor, float, int],
# map_cond=None,
# env_cond=None
sample = torch.randn(B, 32, 2).to(device)
timestep = torch.tensor([1]*B).to(device)
map_cond = torch.randn(B, 3, 8, 8).float().to(device)
env_cond = torch.randint(0, 7, (B, 4)).float().to(device)

stats = summary(
    net,
    input_data={'sample': sample, 'timestep': timestep, 'map_cond': map_cond, 'env_cond': env_cond},
    depth=1,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("output_size", "num_params", "params_percent", "mult_adds"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                                      Output Shape              Param #                   Param %                   Mult-Adds
MultiHypothesisTrajectoryPlanner (MultiHypothesisTrajectoryPlanner)    [64, 4, 32, 2]            --                             --                   --
├─ViT (map_encoder): 1-1                                               [64, 64, 8, 8]            14,883,120                 17.89%                   1,466,483,712
├─OBSEmbedEncoder (env_encoder): 1-2                                   [64, 64]                  16,896                      0.02%                   1,081,344
├─MultiHypothesisDiffusionTrajDecoder (diff_decoder): 1-3              [64, 4, 32, 2]            68,275,842                 82.09%                   111,670,284,672
Total params: 83,175,858
Trainable params: 83,175,858
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 113.14
Input size (MB): 0.07
Forward/backward pass size (MB): 496.31
Params size (MB): 33

#### Diffuser V2 + BiLatent + Multi-hypothesis Refiner

In [15]:
import torch
from core.diffuser.networks.traj_planner.BiLatentMultiHypoPlanner import BiLatentMultiHypoPlanner # Model Class
from core.diffuser.config.diffuser_bil_mhypo import BiLatentMultiHypoConfig # Configuration Class

In [16]:
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
traj_planner_config = BiLatentMultiHypoConfig()


traj_planner_dict = traj_planner_config.to_dict()
traj_planner_net_config = traj_planner_dict['network_config']

device='cuda'

net = BiLatentMultiHypoPlanner(traj_planner_net_config)
noise_scheduler = DDPMScheduler(
            num_train_timesteps=100,
            beta_schedule='squaredcos_cap_v2',
            clip_sample=True,
            prediction_type='epsilon')
net.set_noise_scheduler(noise_scheduler)
net = net.to(device)

In [17]:
from torchinfo import summary
B=64

# sample: torch.Tensor,
# timestep: Union[torch.Tensor, float, int],
# map_cond=None,
# env_cond=None
sample = torch.randn(B, 32, 2).to(device)
timestep = torch.tensor([1]*B).to(device)
map_cond = torch.randn(B, 3, 8, 8).float().to(device)
env_cond = torch.randint(0, 7, (B, 4)).float().to(device)

stats = summary(
    net,
    input_data={'sample': sample, 'timestep': timestep, 'map_cond': map_cond, 'env_cond': env_cond},
    depth=1,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("output_size", "num_params", "params_percent", "mult_adds"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                                 Output Shape              Param #                   Param %                   Mult-Adds
BiLatentMultiHypoPlanner (BiLatentMultiHypoPlanner)               [64, 4, 32, 2]            --                             --                   --
├─ViT (map_encoder): 1-1                                          [64, 64, 8, 8]            14,883,120                 17.83%                   1,466,483,712
├─OBSEmbedEncoder (env_encoder): 1-2                              [64, 64]                  16,896                      0.02%                   1,081,344
├─BiLatentMultiHypoDiffTrajDecoder (diff_decoder): 1-3            [64, 4, 32, 2]            68,580,932                 82.15%                   93,848,016,128
Total params: 83,480,948
Trainable params: 83,480,948
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 95.32
Input size (MB): 0.07
Forward/backward pass size (MB): 725.03
Params size (MB): 332.85
Estimated Total Size (

### Diffuser V2 + Multi-Resolution based Refiner

In [21]:
import torch
from core.diffuser.networks.traj_planner.MultiResTrajPlanner import MultiResTrajPlanner  # Model Class
from core.diffuser.config.diffuser_mres import MultiResTrajectoryPlannerConfig # Configuration Class

In [22]:
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
traj_planner_config = MultiResTrajectoryPlannerConfig()


traj_planner_dict = traj_planner_config.to_dict()
traj_planner_net_config = traj_planner_dict['network_config']

device='cuda'

net = MultiResTrajPlanner(traj_planner_net_config)
noise_scheduler = DDPMScheduler(
            num_train_timesteps=100,
            beta_schedule='squaredcos_cap_v2',
            clip_sample=True,
            prediction_type='epsilon')
net.set_noise_scheduler(noise_scheduler)
net = net.to(device)

In [23]:
from torchinfo import summary
B=64

# sample: torch.Tensor,
# timestep: Union[torch.Tensor, float, int],
# map_cond=None,
# env_cond=None
sample = torch.randn(B, 32, 2).to(device)
timestep = torch.tensor([1]*B).to(device)
map_cond = torch.randn(B, 3, 8, 8).float().to(device)
env_cond = torch.randint(0, 7, (B, 4)).float().to(device)

stats = summary(
    net,
    input_data={'sample': sample, 'timestep': timestep, 'map_cond': map_cond, 'env_cond': env_cond},
    depth=1,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("output_size", "num_params", "params_percent", "mult_adds"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                                 Output Shape              Param #                   Param %                   Mult-Adds
MultiResTrajPlanner (MultiResTrajPlanner)                         [64, 32, 2]               --                             --                   --
├─ViT (map_encoder): 1-1                                          [64, 64, 8, 8]            14,883,120                 17.93%                   1,466,483,712
├─OBSEmbedEncoder (env_encoder): 1-2                              [64, 64]                  16,896                      0.02%                   1,081,344
├─MultiResDiffusionDecoder (diff_decoder): 1-3                    [64, 32, 2]               68,110,214                 82.05%                   33,974,555,008
Total params: 83,010,230
Trainable params: 83,010,230
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 35.44
Input size (MB): 0.07
Forward/backward pass size (MB): 388.63
Params size (MB): 332.03
Estimated Total Size (

### Potential-based Diffusion Model Analysis

In [1]:
from core.pb_diffusion.configs.pb_diffusion_cfg import PBDiffusionConfig
from core.pb_diffusion.networks.pb_traj_planner import PBTrajPlanner
from core.pb_diffusion.diffusion.diffusion_pb import GaussianDiffusionPB

def build_pb_diff_from_cfg(model_id):
    if model_id == 0:
        config = PBDiffusionConfig()
        config_dict = config.to_dict()
        config_dict['diffusion_config']['clip_denoised'] = True
        model = PBTrajPlanner(config_dict['network_config'])
        
        diffusion_config = config_dict['diffusion_config']
        diffusion_model = GaussianDiffusionPB(model=model, **diffusion_config)
        
    return model, diffusion_model, config_dict

In [2]:
device='cuda:7'
model, diffusion_model, config_dict = build_pb_diff_from_cfg(0)
diffusion_model.to(device)
model.to(device)

[ models/temporal_cond ] Channel dimensions: [(2, 96), (96, 384), (384, 768)] 
[ models/temporal_cond ] conv_zero_init L2 False
[TemporalUnet_WCond] concept_drop_prob: 0.2
[TemporalUnet_WCond] time_dim: 224
[Unet down_times] 3 
[Luo self.train_apply_condition] True
[Luo self.set_cond_noise_to_0] False
[set manual loss weight] 0 0.0
[set manual loss weight] -1 0.0


PBTrajPlanner(
  (map_encoder): ViT(
    (stem_conv): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (3): ReLU()
    )
    (to_patch_embedding): Sequential(
      (0): Patchfier(
        (unfolder): Unfold(kernel_size=(4, 4), dilation=1, padding=0, stride=(4, 4))
      )
      (1): Linear(in_features=512, out_features=2048, bias=True)
      (2): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (3): ReLU()
      (4): Linear(in_features=2048, out_features=512, bias=True)
      (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (patch_to_featmap): UnPatchfier()
    (dropout): Dropout(p=0.1, inplace=False)
    (transformer): TransformerEncoder(
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): SelfAttention(
            (norm): LayerN

In [3]:
import torch
from torchinfo import summary
B=64

# sample: torch.Tensor,
# timestep: Union[torch.Tensor, float, int],
# map_cond=None,
# env_cond=None
sample = torch.randn(B, 32, 2).to(device)
timestep = torch.tensor([1]*B).to(device)
map_cond = torch.randn(B, 3, 8, 8).float().to(device)
obs_cond = torch.randint(0, 7, (B, 4)).float().to(device)

# sample = self.diffusion_model(map_cond, obs_cond, return_diffusion=return_diffusion, use_ddim=self.use_ddim)

# stats = summary(
#     diffusion_model,
#     input_data={'map_cond': map_cond, 'obs_cond': obs_cond, 'return_diffusion': True, 'use_ddim': True},
#     depth=1,                      # increase to show nested submodules
#     verbose=1,                     # 2 = even more detail
#     col_names=("output_size", "num_params", "params_percent", "mult_adds"),
#     row_settings=("var_names", "depth"),
# )

# def forward(self, sample, timestep, map_cond, obs_cond,
                # use_dropout=True, force_dropout=False, half_fd=False):

# stats = summary(
#     model,
#     input_data={'sample': sample, 'timestep': timestep, 'map_cond': map_cond, 'obs_cond': obs_cond, 'use_dropout': False, 'force_dropout': False, 'half_fd': False},
#     depth=1,                      # increase to show nested submodules
#     verbose=1,                     # 2 = even more detail
#     col_names=("output_size", "num_params", "params_percent", "mult_adds"),
#     row_settings=("var_names", "depth"),
# )

stats = summary(
    model,
    depth=1,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("num_params", "params_percent"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                            Param #                   Param %
PBTrajPlanner (PBTrajPlanner)                                --                             --
├─ViT (map_encoder): 1-1                                     14,883,120                 30.63%
├─OBSEmbedEncoder (obs_encoder): 1-2                         16,896                      0.03%
├─TemporalUnet_WCond (diff_decoder): 1-3                     33,690,434                 69.34%
Total params: 48,590,450
Trainable params: 48,590,450
Non-trainable params: 0


### Potential Based Diffusion Model (PBDMP V1) + BiLatent Model

In [27]:
from core.pb_diffusion.configs.pb_diffusion_bil_cfg import PBDiffusionBILConfig
from core.pb_diffusion.networks.pb_bil_traj_planner import PBTrajBILPlanner
from core.pb_diffusion.diffusion.diffusion_pb import GaussianDiffusionPB

def build_pb_diff_from_cfg(model_id):
    if model_id == 0:
        config = PBDiffusionBILConfig()
        config_dict = config.to_dict()
        config_dict['diffusion_config']['clip_denoised'] = True
        model = PBTrajBILPlanner(config_dict['network_config'])
        
        diffusion_config = config_dict['diffusion_config']
        diffusion_model = GaussianDiffusionPB(model=model, **diffusion_config)
        
    return model, diffusion_model, config_dict

In [28]:
device='cuda:7'
model, diffusion_model, config_dict = build_pb_diff_from_cfg(0)
diffusion_model.to(device)
model.to(device)

[ models/temporal_cond ] Channel dimensions: [(2, 128), (128, 512), (512, 1024)] 
[ models/temporal_cond ] conv_zero_init L2 False
[TemporalUnet_WCond] concept_drop_prob: 0.2
[TemporalUnet_WCond] time_dim: 256
[Unet down_times] 3 
[Luo self.train_apply_condition] True
[Luo self.set_cond_noise_to_0] False
[set manual loss weight] 0 0.0
[set manual loss weight] -1 0.0


PBTrajBILPlanner(
  (map_encoder): ViT(
    (stem_conv): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (3): ReLU()
    )
    (to_patch_embedding): Sequential(
      (0): Patchfier(
        (unfolder): Unfold(kernel_size=(4, 4), dilation=1, padding=0, stride=(4, 4))
      )
      (1): Linear(in_features=512, out_features=2048, bias=True)
      (2): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (3): ReLU()
      (4): Linear(in_features=2048, out_features=512, bias=True)
      (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (patch_to_featmap): UnPatchfier()
    (dropout): Dropout(p=0.1, inplace=False)
    (transformer): TransformerEncoder(
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): SelfAttention(
            (norm): Lay

In [29]:
from torchinfo import summary

stats = summary(
    model,
    depth=2,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("num_params", "params_percent"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                            Param #                   Param %
PBTrajBILPlanner (PBTrajBILPlanner)                          --                             --
├─ViT (map_encoder): 1-1                                     3,072                       0.00%
│    └─Sequential (stem_conv): 2-1                           10,144                      0.01%
│    └─Sequential (to_patch_embedding): 2-2                  2,104,832                   2.81%
│    └─UnPatchfier (patch_to_featmap): 2-3                   --                             --
│    └─Dropout (dropout): 2-4                                --                             --
│    └─TransformerEncoder (transformer): 2-5                 12,608,512                 16.86%
│    └─ViT2ConvBridge (vit2conv_bridge): 2-6                 119,632                     0.16%
│    └─Linear (mlp_head): 2-7                                36,928                      0.05%
├─OBSEmbedEncoder (obs_encoder): 1-2              

### Potential-Based Diffusion Model V2

In [1]:
# from core.pb_diffusion.configs.pb_diffusion_v2_cfg import PBDiffusionV2Config
from core.pb_diffusion.configs.pb_ddp_v2_cfg import PBV2DDPConfig
from core.pb_diffusion.networks.pb_traj_planner_v2 import PBTrajPlannerV2
from core.pb_diffusion.diffusion.diffusion_pb import GaussianDiffusionPB

def build_pb_diff_from_cfg():
    
    # config = PBDiffusionV2Config()
    config = PBV2DDPConfig()
    config_dict = config.to_dict()
    config_dict['diffusion_config']['clip_denoised'] = True
    model = PBTrajPlannerV2(config_dict['network_config'])
    
    diffusion_config = config_dict['diffusion_config']
    diffusion_model = GaussianDiffusionPB(model=model, **diffusion_config)
    
    return model, diffusion_model, config_dict

In [2]:
device='cuda:7'
model, diffusion_model, config_dict = build_pb_diff_from_cfg()
diffusion_model.to(device)
model.to(device)

[ models/temporal_cond ] Channel dimensions: [(2, 64), (64, 128), (128, 256), (256, 512)] 
[ models/temporal_cond ] conv_zero_init L2 False
[TemporalUnet_WCond] concept_drop_prob: 0.2
[TemporalUnet_WCond] time_dim: 256
[Unet down_times] 3 
[Luo self.train_apply_condition] True
[Luo self.set_cond_noise_to_0] False
[set manual loss weight] 0 0.0
[set manual loss weight] -1 0.0


PBTrajPlannerV2(
  (map_encoder): ViT(
    (stem_conv): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (3): ReLU()
    )
    (to_patch_embedding): Sequential(
      (0): Patchfier(
        (unfolder): Unfold(kernel_size=(4, 4), dilation=1, padding=0, stride=(4, 4))
      )
      (1): Linear(in_features=512, out_features=2048, bias=True)
      (2): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (3): ReLU()
      (4): Linear(in_features=2048, out_features=512, bias=True)
      (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (patch_to_featmap): UnPatchfier()
    (dropout): Dropout(p=0.1, inplace=False)
    (transformer): TransformerEncoder(
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): SelfAttention(
            (norm): Laye

In [3]:
from torchinfo import summary

stats = summary(
    model,
    depth=2,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("num_params", "params_percent"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                            Param #                   Param %
PBTrajPlannerV2 (PBTrajPlannerV2)                            --                             --
├─ViT (map_encoder): 1-1                                     9,216                       0.02%
│    └─Sequential (stem_conv): 2-1                           9,568                       0.02%
│    └─Sequential (to_patch_embedding): 2-2                  2,104,832                   3.97%
│    └─UnPatchfier (patch_to_featmap): 2-3                   --                             --
│    └─Dropout (dropout): 2-4                                --                             --
│    └─TransformerEncoder (transformer): 2-5                 12,608,512                 23.81%
│    └─ViT2ConvBridge (vit2conv_bridge): 2-6                 121,744                     0.23%
│    └─Linear (mlp_head): 2-7                                82,048                      0.15%
├─OBSEmbedEncoder (obs_encoder): 1-2              

### Run Composite Diffusion Models

In [1]:
from core.comp_diffusion.configs.comp_diffusion_cfg import CompDiffusionConfig
from core.comp_diffusion.cd_stgl_sml_dfu.stgl_sml_traj_planner import CompTrajPlanner
from core.comp_diffusion.cd_stgl_sml_dfu.stgl_sml_diffusion_v1 import Stgl_Sml_GauDiffusion_InvDyn_V1

def build_comp_diff_from_cfg(model_id):
    if model_id == 0:
        config = CompDiffusionConfig()
        config_dict = config.to_dict()
        config_dict['diffusion_config']['clip_denoised'] = True
        model = CompTrajPlanner(config_dict['network_config'])
        
        diffusion_config = config_dict['diffusion_config']
        diffusion_model = Stgl_Sml_GauDiffusion_InvDyn_V1(model=model, **diffusion_config)
        
    return model, diffusion_model, config_dict

In [2]:
device='cuda:7'
model, diffusion_model, config_dict = build_comp_diff_from_cfg(0)
diffusion_model.to(device)
model.to(device)

[ models/Unet1D_TjTi_Stgl_Cond_V1 ] Channel dimensions: [(2, 128), (128, 512)] 
[ models/Traj_Time_Encoder ] Channel dimensions: [(2, 64), (64, 256)] 
TjTi Enc: self.cnn_out_dim=128, self.final_mlp_dims=[1280, 512, 128] 
TjTi Enc: unet:  [(2, 64), (64, 256)] 
TjTi Enc: mid_dim=256, self.last_hzn=2, self.f_mlp_in_dim=256 
TjTi Enc: self.f_mlp_in_dim=256, f_mlp_in_outs=[(256, 1280), (1280, 512), (512, 128)] 
[ models/Traj_Time_Encoder ] Channel dimensions: [(2, 64), (64, 256)] 
TjTi Enc: self.cnn_out_dim=128, self.final_mlp_dims=[1280, 512, 128] 
TjTi Enc: unet:  [(2, 64), (64, 256)] 
TjTi Enc: mid_dim=256, self.last_hzn=2, self.f_mlp_in_dim=256 
TjTi Enc: self.f_mlp_in_dim=256, f_mlp_in_outs=[(256, 1280), (1280, 512), (512, 128)] 
[TemporalUnet_WCond] time_dim=256, tot_cond_dim=704, self.ext_cond_dim=256 
[TemporalUnet_WCond]: in_out:  [(2, 128), (128, 512)]
[Unet down_times] 3 
self.diff_config['tr_1side_drop_prob']=0.2 
self.tr_no_ovlp_none=False 


CompTrajPlanner(
  (map_encoder): ViT(
    (stem_conv): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (3): ReLU()
    )
    (to_patch_embedding): Sequential(
      (0): Patchfier(
        (unfolder): Unfold(kernel_size=(4, 4), dilation=1, padding=0, stride=(4, 4))
      )
      (1): Linear(in_features=512, out_features=2048, bias=True)
      (2): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (3): ReLU()
      (4): Linear(in_features=2048, out_features=512, bias=True)
      (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (patch_to_featmap): UnPatchfier()
    (dropout): Dropout(p=0.1, inplace=False)
    (transformer): TransformerEncoder(
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): SelfAttention(
            (norm): Laye

In [3]:
from torchinfo import summary

stats = summary(
    model,
    depth=2,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("num_params", "params_percent"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                                 Param #                   Param %
CompTrajPlanner (CompTrajPlanner)                                 --                             --
├─ViT (map_encoder): 1-1                                          3,072                       0.01%
│    └─Sequential (stem_conv): 2-1                                10,144                      0.02%
│    └─Sequential (to_patch_embedding): 2-2                       2,104,832                   4.58%
│    └─UnPatchfier (patch_to_featmap): 2-3                        --                             --
│    └─Dropout (dropout): 2-4                                     --                             --
│    └─TransformerEncoder (transformer): 2-5                      12,608,512                 27.42%
│    └─ViT2ConvBridge (vit2conv_bridge): 2-6                      119,632                     0.26%
│    └─Linear (mlp_head): 2-7                                     36,928                      0.08%


### Run Composite Diffusion V2

In [1]:
# from core.comp_diffusion.configs.comp_diffusion_cfg_v2 import CompDiffusionConfigV2
from core.comp_diffusion.configs.comp_v2_ddp_cfg import CompDDPV2Config
from core.comp_diffusion.cd_stgl_sml_dfu.stgl_sml_traj_planner_v2 import CompTrajPlannerV2
from core.comp_diffusion.cd_stgl_sml_dfu.stgl_sml_diffusion_v1 import Stgl_Sml_GauDiffusion_InvDyn_V1

def build_comp_diff_from_cfg():
    
    # config = CompDiffusionConfigV2()
    config = CompDDPV2Config()
    config_dict = config.to_dict()
    config_dict['diffusion_config']['clip_denoised'] = True
    model = CompTrajPlannerV2(config_dict['network_config'])
    
    diffusion_config = config_dict['diffusion_config']
    diffusion_model = Stgl_Sml_GauDiffusion_InvDyn_V1(model=model, **diffusion_config)
        
    return model, diffusion_model, config_dict

In [2]:
device='cuda:7'
model, diffusion_model, config_dict = build_comp_diff_from_cfg()
diffusion_model.to(device)
model.to(device)

[ models/Unet1D_TjTi_Stgl_Cond_V1 ] Channel dimensions: [(2, 64), (64, 192), (192, 384)] 
[ models/Traj_Time_Encoder ] Channel dimensions: [(2, 64), (64, 128), (128, 256)] 
TjTi Enc: self.cnn_out_dim=128, self.final_mlp_dims=[1280, 512, 256] 
TjTi Enc: unet:  [(2, 64), (64, 128), (128, 256)] 
TjTi Enc: mid_dim=256, self.last_hzn=2, self.f_mlp_in_dim=256 
TjTi Enc: self.f_mlp_in_dim=256, f_mlp_in_outs=[(256, 1280), (1280, 512), (512, 256)] 
[ models/Traj_Time_Encoder ] Channel dimensions: [(2, 64), (64, 128), (128, 256)] 
TjTi Enc: self.cnn_out_dim=128, self.final_mlp_dims=[1280, 512, 256] 
TjTi Enc: unet:  [(2, 64), (64, 128), (128, 256)] 
TjTi Enc: mid_dim=256, self.last_hzn=2, self.f_mlp_in_dim=256 
TjTi Enc: self.f_mlp_in_dim=256, f_mlp_in_outs=[(256, 1280), (1280, 512), (512, 256)] 
[TemporalUnet_WCond] time_dim=256, tot_cond_dim=960, self.ext_cond_dim=512 
[TemporalUnet_WCond]: in_out:  [(2, 64), (64, 192), (192, 384)]
[Unet down_times] 3 
self.diff_config['tr_1side_drop_prob']=0.

CompTrajPlannerV2(
  (map_encoder): ViT(
    (stem_conv): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (3): ReLU()
    )
    (to_patch_embedding): Sequential(
      (0): Patchfier(
        (unfolder): Unfold(kernel_size=(4, 4), dilation=1, padding=0, stride=(4, 4))
      )
      (1): Linear(in_features=512, out_features=2048, bias=True)
      (2): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (3): ReLU()
      (4): Linear(in_features=2048, out_features=512, bias=True)
      (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (patch_to_featmap): UnPatchfier()
    (dropout): Dropout(p=0.1, inplace=False)
    (transformer): TransformerEncoder(
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): SelfAttention(
            (norm): La

In [3]:
from torchinfo import summary

stats = summary(
    model,
    depth=2,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("num_params", "params_percent"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                                      Param #                   Param %
CompTrajPlannerV2 (CompTrajPlannerV2)                                  --                             --
├─ViT (map_encoder): 1-1                                               9,216                       0.02%
│    └─Sequential (stem_conv): 2-1                                     9,568                       0.02%
│    └─Sequential (to_patch_embedding): 2-2                            2,104,832                   3.64%
│    └─UnPatchfier (patch_to_featmap): 2-3                             --                             --
│    └─Dropout (dropout): 2-4                                          --                             --
│    └─TransformerEncoder (transformer): 2-5                           12,608,512                 21.82%
│    └─ViT2ConvBridge (vit2conv_bridge): 2-6                           121,744                     0.21%
│    └─Linear (mlp_head): 2-7                          

### ReDiffuser

In [3]:
# from core.rediffuser.configs.rediff_cfg import ReDiffConfig
from core.rediffuser.configs.rediff_ddp_cfg import ReDiffDDPConfig
from core.rediffuser.networks.traj_planner import RediffuserPlanner
from core.rediffuser.networks.diffuser.models.diffusion import GaussianDiffusion

def build_rediff_from_cfg():
    
    # config = ReDiffConfig()
    config = ReDiffDDPConfig()
    config_dict = config.to_dict()
    config_dict['diffusion_config']['clip_denoised'] = True
    model = RediffuserPlanner(config_dict['network_config'])
    
    diffusion_config = config_dict['diffusion_config']
    diffusion_model = GaussianDiffusion(model=model, **diffusion_config)
        
    return model, diffusion_model, config_dict

In [4]:
device='cuda:1'
model, diffusion_model, config_dict = build_rediff_from_cfg()
diffusion_model.to(device)
model.to(device)

[ models/temporal ] Channel dimensions: [(2, 64), (64, 128), (128, 256), (256, 512)]
[(2, 64), (64, 128), (128, 256), (256, 512)]


RediffuserPlanner(
  (map_encoder): ViT(
    (stem_conv): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
      (3): ReLU()
    )
    (to_patch_embedding): Sequential(
      (0): Patchfier(
        (unfolder): Unfold(kernel_size=(4, 4), dilation=1, padding=0, stride=(4, 4))
      )
      (1): Linear(in_features=512, out_features=2048, bias=True)
      (2): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (3): ReLU()
      (4): Linear(in_features=2048, out_features=512, bias=True)
      (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (patch_to_featmap): UnPatchfier()
    (dropout): Dropout(p=0.1, inplace=False)
    (transformer): TransformerEncoder(
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): SelfAttention(
            (norm): La

In [5]:
from torchinfo import summary

stats = summary(
    model,
    depth=2,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("num_params", "params_percent"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                            Param #                   Param %
RediffuserPlanner (RediffuserPlanner)                        --                             --
├─ViT (map_encoder): 1-1                                     9,216                       0.02%
│    └─Sequential (stem_conv): 2-1                           10,144                      0.02%
│    └─Sequential (to_patch_embedding): 2-2                  2,104,832                   3.97%
│    └─UnPatchfier (patch_to_featmap): 2-3                   --                             --
│    └─Dropout (dropout): 2-4                                --                             --
│    └─TransformerEncoder (transformer): 2-5                 12,608,512                 23.81%
│    └─ViT2ConvBridge (vit2conv_bridge): 2-6                 121,744                     0.23%
│    └─Linear (mlp_head): 2-7                                82,048                      0.15%
├─OBSEmbedEncoder (obs_encoder): 1-2              

### Diffuser SimFusion

In [1]:
from core.diffuser.config.diffuser_sim_fusion import SimFusionTrajPlannerConfig
from core.diffuser.networks.traj_planner.SimFusionTrajPlanner import SimFusionTrajPlanner

In [2]:
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
traj_planner_config = SimFusionTrajPlannerConfig()


traj_planner_dict = traj_planner_config.to_dict()
traj_planner_net_config = traj_planner_dict['network_config']

device='cuda'

net = SimFusionTrajPlanner(traj_planner_net_config)
noise_scheduler = DDPMScheduler(
            num_train_timesteps=100,
            beta_schedule='squaredcos_cap_v2',
            clip_sample=True,
            prediction_type='epsilon')
net.set_noise_scheduler(noise_scheduler)
net = net.to(device)

In [3]:
import torch
from torchinfo import summary
B=64
# sample: torch.Tensor,
# timestep: Union[torch.Tensor, float, int],
# map_cond=None,
# env_cond=None
sample = torch.randn(B, 32, 2).to(device)
timestep = torch.tensor([1]*B).long().to(device)
map_cond = torch.randn(B, 3, 8, 8).float().to(device)
env_cond = torch.randint(0, 7, (B, 4)).float().to(device)

stats = summary(
    net,
    input_data={'sample': sample, 'timestep': timestep, 'map_cond': map_cond, 'env_cond': env_cond},
    depth=2,                      # increase to show nested submodules
    verbose=1,                     # 2 = even more detail
    col_names=("output_size", "num_params", "params_percent", "mult_adds"),
    row_settings=("var_names", "depth"),
)

Layer (type (var_name):depth-idx)                                 Output Shape              Param #                   Param %                   Mult-Adds
SimFusionTrajPlanner (SimFusionTrajPlanner)                       [64, 32, 2]               --                             --                   --
├─ViT (map_encoder): 1-1                                          [64, 64, 8, 8]            3,072                       0.01%                   --
│    └─Sequential (stem_conv): 2-1                                [64, 32, 8, 8]            10,144                      0.02%                   41,549,824
│    └─Sequential (to_patch_embedding): 2-2                       [64, 4, 512]              2,104,832                   4.55%                   134,709,248
│    └─Dropout (dropout): 2-3                                     [64, 5, 512]              --                             --                   --
│    └─TransformerEncoder (transformer): 2-4                      [64, 5, 512]              12